# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step demonstration for loading and exploring the FAIR² dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) using the `mlcroissant` library.

### Dataset Source
The dataset is described and distributed using a [Croissant schema](https://mlcommons.org/croissant/) available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We'll print the name and description of the dataset as specified in its schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata  # This is a mlcroissant Metadata object
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review available record sets and their fields by `@id`. We'll list all record sets and fields contained in the dataset with their unique `@id` values, which will be used in subsequent steps.

In [ ]:
# List all record sets by @id and fields by their @id
print('Record Sets:')
record_sets = dataset.record_sets  # this is a dict mapping @id -> RecordSet
for record_set_id, record_set in record_sets.items():
    print(f"- RecordSet @id: {record_set_id}, name: {getattr(record_set, 'name', '<no name>')}")
    if hasattr(record_set, 'fields'):
        for field_id, field in record_set.fields.items():
            print(f"    - Field @id: {field_id}, name: {getattr(field, 'name', '<no name>')}, dataType: {getattr(field, 'data_type', '<no type>')}")

## 3. Data Extraction

Let's extract data from the available record set(s) for analysis. We will use the record set and field `@id`s listed above.
We'll load the main tabular data into DataFrame(s), using the unique `@id` for each record set.

**Note:** If your dataset includes multiple record sets, they will all be loaded into separate DataFrames indexed by their `@id`.

In [ ]:
dataframes = {}
## Collect all record set @id's
record_set_ids = list(dataset.record_sets.keys())
print(f'Record sets found: {record_set_ids}')

# Load all available record sets into DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for record set {record_set_id}.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for record set {record_set_id}")

# Choose the primary record set for further exploration (use the first listed if unsure)
if len(dataframes):
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No dataframes loaded. Please check available record sets.')

## 4. Exploratory Data Analysis (EDA)

Apply core data processing steps: filtering, normalization, and grouping by key variables. Please ensure you select appropriate field `@id`s from the earlier overview.

- We'll select a numeric field (e.g., patient age, id, or interval) and a grouping field (e.g., anatomical site, MSI status, or gender) for example operations.
- Remember to reference fields by their `@id`, not by the display name.

In [ ]:
main_df = dataframes[main_record_set_id]

# List candidate numeric and grouping fields by @id and data type
print('Available fields for EDA:')
field_ids = []
group_candidates = []
for record_set in dataset.record_sets.values():
    for field_id, field in record_set.fields.items():
        field_ids.append((field_id, getattr(field, 'name', '<no name>'), getattr(field, 'data_type', '<no type>')))
        if getattr(field, 'data_type', '').lower() in {'integer','float','number'}:
            print(f"Numeric: {field_id} - name: {getattr(field, 'name', '<no name>')}")
        else:
            group_candidates.append((field_id, getattr(field, 'name', '<no name>')))
print('\nCandidate grouping fields:')
for x in group_candidates:
    print(f"- {x[0]} - name: {x[1]}")

# ---- Specify your field @id here ----
# Choose a numeric field. (Replace below with a field @id from the printed list, e.g. 'age', or interval, etc.)
numeric_field_id = None
for field_id, field_name, field_type in field_ids:
    if field_type.lower() in {'integer','float','number'} and field_id in main_df.columns:
        numeric_field_id = field_id
        break

# Choose a grouping field. (Replace below with a field @id from the printed candidate grouping fields)
group_field_id = None
for field_id, field_name in group_candidates:
    if field_id in main_df.columns and pd.api.types.is_object_dtype(main_df[field_id]):
        group_field_id = field_id
        break

if not numeric_field_id or not group_field_id:
    print("No suitable numeric or grouping field found. Please review the above lists.")
else:
    print(f"\nUsing numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # Example filtering
    # Choose a threshold value based on field properties. For demo, use 10.
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id].astype(float) > threshold].copy()

    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped aggregation
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization

Let's visualize data distributions or relationships between fields in the dataset. For example, we can plot the distribution of the numeric field, and show average values per group. Adjust field `@id` accordingly as chosen above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,5))
    main_df[numeric_field_id].astype(float).hist(bins=16)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if group_field_id and group_field_id in main_df.columns and numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id].astype(float))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated the use of the `mlcroissant` library to:

- Load and overview a dataset defined by a Croissant schema.
- Enumerate available record sets and fields with unique identifiers.
- Extract tabular data and perform basic exploratory data analysis (EDA), including filtering, normalization, grouping, and visualization based on field `@id`s.

The FAIR² dataset enables investigations into clinicopathological features, the distribution of MSI status, and more for second primary colorectal cancer in cancer survivors. Continue your analysis by leveraging the well-documented schema, record set, and field `@id`s for reproducible, transparent research.